In [21]:
import pandas as pd
import requests 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

plt.style.use('default') # sets matplotlib to use default plotting style
sns.set_context('talk') # Used for clarity in project



In [22]:
#Due to the structure of Met Éireann’s observational data, a subset of representative stations was selected to analyse rainfall and wind trends.

stations = {"Cork Airport": "cork%20airport",
            "Dublin Airport": "dublin%20airport",
            "Shannon Airport": "shannon%20airport",
            "Knock Airport": "knock%20airport",
            "Valentia Observatory": "valentia%20observatory" }

In [23]:
# Fetch and parse rainfall data

def fetch_rainfall_data(station_name, station_url):
    url = f'https://prodapi.metweb.ie/monthly-data/{station_url}'
    response = requests.get(url)
    data = response.json()

    rainfall = data['total_rainfall']['report']

    records =[]

    for year in ['2024', '2025']:
        if year in rainfall:
            for month, value in rainfall[year].items():
                if month != 'annual' and value not in ['','n/a']:
                    records.append({
                        'station': station_name,
                        'year': int(year),
                        'month': month,
                        'rainfall_mm': float(value)

                })
                    
    return pd.DataFrame(records)


In [24]:
# Pull data from all stations

df_list = []

for station_name, station_url in stations.items():
    df_station = fetch_rainfall_data (station_name, station_url)
    df_list.append(df_station)

rainfall_df = pd.concat(df_list, ignore_index=True)

In [25]:
# Inspet the result

rainfall_df.head()

,station,year,month,rainfall_mm
0,Cork Airport,2024,january,99.5
1,Cork Airport,2024,february,157.7
2,Cork Airport,2024,mar,168.2
3,Cork Airport,2024,apr,107.1
4,Cork Airport,2024,may,109.6


In [26]:
# Compare 2024 vs 2025

comparison = (
    rainfall_df
    .groupby(['station', 'year'])['rainfall_mm']
    .mean()
    .reset_index()
)

comparison

,station,year,rainfall_mm
0,Cork Airport,2024,105.833333
1,Cork Airport,2025,114.366667
2,Dublin Airport,2024,55.833333
3,Dublin Airport,2025,67.016667
4,Knock Airport,2024,119.175000
5,Knock Airport,2025,114.583333
6,Shannon Airport,2024,81.150000
7,Shannon Airport,2025,85.666667
8,Valentia Observatory,2024,137.691667
9,Valentia Observatory,2025,150.633333


In [27]:
# Indenify extreme rainfall onths

threshold = rainfall_df['rainfall_mm'].quantile(0.95)

extreme_months = rainfall_df

In [28]:
# Load the Met Eireann Station CSV file
# Used engine python and on bad lines skip as Met Eireann Data coming up with error on loadind file
df = pd.read_csv(
    'StationDetails.csv',
    engine='python',
    on_bad_lines='skip'
)

# Inspect Columns
print(df.shape)
print(df.columns)

(2080, 10)
Index(['county', 'station name', 'name', 'height(m)', 'easting', 'northing',
       'latitude', 'longitude', 'open year', 'close year'],
      dtype='object')
